# Notebook 07 — Local OpenAI-Compatible Serving

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/07_local_openai_compatible_serving.ipynb)

The fastest way to make a local model useful to application code is often an OpenAI-compatible HTTP interface. This notebook builds that idea from first principles with a small local mock and explicit request contracts.

## What you will build

- define request and response schemas for local chat completions
- stand up a notebook-local FastAPI service
- test `/v1/models` and `/v1/chat/completions` end to end
- sketch adapters for `llama-server` and other open runtimes
- add simple telemetry and overload control

## Why compatibility matters

Compatibility lowers friction. If your local runtime speaks a familiar API shape, you can swap between notebook mocks, `llama.cpp`, and higher-throughput engines without rewriting every caller.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

In [ ]:
random.seed(7)

ARTIFACT_DIR = ARTIFACT_ROOT / "07_local_openai_compatible_serving"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_LOG = []

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Define the API contract

We start small: chat completions, a health check, and model discovery. This is enough to wire up many notebook and application experiments.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import httpx
import uvicorn

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str
    messages: list[ChatMessage]
    temperature: float = 0.2
    max_tokens: int = 128
    stream: bool = False

class ResponseMessage(BaseModel):
    role: str = "assistant"
    content: str

class Choice(BaseModel):
    index: int
    message: ResponseMessage
    finish_reason: str = "stop"

class Usage(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int

## Step 2: Create a simple model registry

A registry gives clients something stable to inspect. It also lets you map friendly API names to concrete runtime commands later.

In [ ]:
MODEL_REGISTRY = {
    "llama-local-q4": {
        "backend": "llama.cpp",
        "artifact": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
        "context_length": 8192,
    },
    "qwen-local-q5": {
        "backend": "llama.cpp",
        "artifact": "Qwen2.5-7B-Instruct-Q5_K_M.gguf",
        "context_length": 8192,
    },
}

pd.DataFrame([{**{"model": name}, **config} for name, config in MODEL_REGISTRY.items()])

## Step 3: Build a deterministic local completion function

A deterministic mock keeps the notebook runnable even when no heavy runtime is installed. The same API surface can later call a real local model.

In [ ]:
def approximate_token_count(text):
    return max(1, math.ceil(len(text.split()) * 1.3))

def make_reply(payload):
    last_user = next((message.content for message in reversed(payload.messages) if message.role == "user"), "")
    prefix = "Local open model reply:"
    answer = prefix + " " + last_user[:180]
    prompt_tokens = sum(approximate_token_count(message.content) for message in payload.messages)
    completion_tokens = approximate_token_count(answer)
    return {
        "id": f"chatcmpl-{int(time.time() * 1000)}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": payload.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": answer},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
        },
    }

## Step 4: Wire the FastAPI app

This app keeps the contract small and explicit. It is intentionally transparent so you can see where validation, logging, and backend dispatch belong.

In [ ]:
app = FastAPI(title="Castalia Local OpenAI-Compatible Mock")

@app.get("/health")
async def health():
    return {"status": "ok", "models": sorted(MODEL_REGISTRY)}

@app.get("/v1/models")
async def list_models():
    data = []
    for name in sorted(MODEL_REGISTRY):
        data.append({"id": name, "object": "model", "owned_by": "castalia"})
    return {"object": "list", "data": data}

@app.post("/v1/chat/completions")
async def chat_completions(payload: ChatRequest):
    if payload.model not in MODEL_REGISTRY:
        raise HTTPException(status_code=404, detail=f"Unknown model: {payload.model}")
    response = make_reply(payload)
    REQUEST_LOG.append({
        "ts": time.time(),
        "model": payload.model,
        "prompt_messages": len(payload.messages),
        "prompt_tokens": response["usage"]["prompt_tokens"],
        "completion_tokens": response["usage"]["completion_tokens"],
    })
    return response

## Step 5: Start the service inside the notebook when you want it

The cell below is off by default so the notebook stays lightweight. Flip the flag when you want a live local HTTP endpoint.

In [ ]:
RUN_SERVER = False
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8017
server = None
server_thread = None

if RUN_SERVER:
    config = uvicorn.Config(app, host=SERVER_HOST, port=SERVER_PORT, log_level="warning")
    server = uvicorn.Server(config)

    def run_server():
        asyncio.set_event_loop(asyncio.new_event_loop())
        server.run()

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    time.sleep(1.0)
    print(f"Server running at http://{SERVER_HOST}:{SERVER_PORT}")
else:
    print("Set RUN_SERVER = True to launch the local API server.")

## Step 6: Send a smoke-test request

If the server is not running, we still preview the exact response shape by calling the same local completion function directly.

In [ ]:
sample_payload = {
    "model": "llama-local-q4",
    "messages": [
        {"role": "system", "content": "You are a concise local systems tutor."},
        {"role": "user", "content": "Explain why OpenAI-compatible serving is useful for local inference."},
    ],
    "temperature": 0.2,
    "max_tokens": 96,
}

if RUN_SERVER:
    response = httpx.post(f"http://{SERVER_HOST}:{SERVER_PORT}/v1/chat/completions", json=sample_payload, timeout=10)
    print("Status:", response.status_code)
    print(json.dumps(response.json(), indent=2))
else:
    preview = make_reply(ChatRequest(**sample_payload))
    print(json.dumps(preview, indent=2))

## Step 7: Preview streaming-compatible chunks

Many local servers stream server-sent events. Even if you do not implement real streaming yet, it helps to see the response shape.

In [ ]:
def stream_chunks(payload_dict):
    payload = ChatRequest(**payload_dict)
    response = make_reply(payload)
    text = response["choices"][0]["message"]["content"]
    emitted = []
    for index, token in enumerate(text.split()[:12]):
        emitted.append({
            "id": response["id"],
            "object": "chat.completion.chunk",
            "choices": [{"index": 0, "delta": {"content": token + " "}, "finish_reason": None}],
            "seq": index,
        })
    emitted.append({
        "id": response["id"],
        "object": "chat.completion.chunk",
        "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
        "seq": len(emitted),
    })
    return emitted

stream_preview = stream_chunks(sample_payload)
stream_preview[:4]

## Step 8: Sketch adapters for real local backends

The same contract can point at different engines. Here we keep backend commands explicit so you can inspect exactly what would run.

In [ ]:
backend_commands = {
    "llama.cpp": [
        "llama-server",
        "-m",
        str(CACHE_DIR / "gguf" / "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"),
        "--host", "127.0.0.1",
        "--port", "8000",
        "-c", "8192",
        "-ngl", "20",
    ],
    "vllm": [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-3B-Instruct",
        "--host", "127.0.0.1",
        "--port", "8001",
    ],
}

for backend, command in backend_commands.items():
    print(backend + ":")
    print(" ".join(command))
    print()

## Step 9: Add simple telemetry

Serving is not only generation. We also want visibility into volume, token load, and model usage.

In [ ]:
def summarize_requests(request_log):
    if not request_log:
        return pd.DataFrame(columns=["metric", "value"])
    total_requests = len(request_log)
    prompt_tokens = sum(item["prompt_tokens"] for item in request_log)
    completion_tokens = sum(item["completion_tokens"] for item in request_log)
    by_model = Counter(item["model"] for item in request_log)
    rows = [
        {"metric": "total_requests", "value": total_requests},
        {"metric": "prompt_tokens", "value": prompt_tokens},
        {"metric": "completion_tokens", "value": completion_tokens},
        {"metric": "top_model", "value": by_model.most_common(1)[0][0]},
    ]
    return pd.DataFrame(rows)

telemetry_df = summarize_requests(REQUEST_LOG)
telemetry_df

## Step 10: Simulate queue admission and overload behavior

A local server on a laptop should refuse overload cleanly instead of pretending every request can run immediately.

In [ ]:
def admit_request(queue_depth, max_queue=4):
    if queue_depth >= max_queue:
        return {"decision": "reject", "status_code": 429, "reason": "queue_full"}
    if queue_depth >= max_queue - 1:
        return {"decision": "degrade", "status_code": 200, "reason": "short_answer_mode"}
    return {"decision": "accept", "status_code": 200, "reason": "normal"}

queue_rows = []
for queue_depth in range(7):
    queue_rows.append({"queue_depth": queue_depth, **admit_request(queue_depth)})

queue_df = pd.DataFrame(queue_rows)
queue_df

## Step 11: Checklist for local compatibility testing

Before calling a local runtime production-like, validate these basics:

- `/health` returns quickly
- `/v1/models` exposes the names clients expect
- `/v1/chat/completions` preserves the OpenAI-style envelope
- unknown models return explicit errors
- overload paths are measured and documented

In [ ]:
manifest = {
    "notebook": "07_local_openai_compatible_serving",
    "models": MODEL_REGISTRY,
    "backend_commands": backend_commands,
    "queue_policy": queue_rows,
    "sample_request": sample_payload,
}

manifest_path = ARTIFACT_DIR / "local_openai_compat_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote manifest:", manifest_path.resolve())

## Recap

You now have a notebook-first serving contract that can point at a mock backend today and a real open-source runtime later. The next notebook compares that kind of service to a raw `transformers` baseline so the benefits of optimized runtimes are visible.